<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</center>
</figure>

# **Universidad de Buenos Aires**
## **Facultad de Ciencias Económicas**

### **Taller de Programación para el Análisis de Datos**

### Prerequisito: cargar la base de datos

Antes de abrir esta notebook, importá el archivo `northwind_mysql.sql` en MySQL.

---

#### 🪟 Windows — MySQL Installer (instalación nativa)

**Opción A — MySQL Command Line Client** (recomendada, viene con MySQL Installer):

1. Abrí el menú Inicio → buscá **MySQL 8.0 Command Line Client** y ejecutalo
2. Te va a pedir la contraseña de root → ingresála
3. Ya estás dentro del shell de MySQL (vas a ver el prompt `mysql>`)
4. Ejecutá el siguiente comando (ajustá la ruta a donde tengas el archivo):
```sql
source C:/Users/TuUsuario/Desktop/northwind_mysql.sql
```
> 💡 Usá barras `/` en la ruta, no `\`. Ese es el error más común en Windows.

**Opción B — MySQL Workbench** (interfaz gráfica):

1. Abrí MySQL Workbench → conectate a tu servidor local
2. Menú **File → Open SQL Script** → seleccioná `northwind_mysql.sql`
3. Presioná el botón ⚡ (Execute) o `Ctrl+Shift+Enter`

**Opción C — Símbolo del sistema (cmd)**:

1. Abrí `cmd` como administrador
2. Navegá a la carpeta donde está el archivo:
```bat
cd C:\Users\TuUsuario\Desktop
```
3. Ejecutá (todo en una línea):
```bat
"C:\Program Files\MySQL\MySQL Server 8.0\bin\mysql.exe" -u root -p < northwind_mysql.sql
```

---

#### 🐧 Linux / macOS

```bash
mysql -u root -p < northwind_mysql.sql
```

---

#### ✅ Verificación (cualquier sistema)

Una vez importado, verificá que las tablas existen:
```sql
USE northwind;
SHOW TABLES;
```
Deberías ver 8 tablas: Categories, Customers, Employees, OrderDetails, Orders, Products, Shippers, Suppliers.

---
## PARTE 0 — Conexión a MySQL

Usamos `mysql-connector-python` para conectarnos y `pandas` para leer los resultados como DataFrames.

```bash
pip install mysql-connector-python pandas
```

In [1]:
import mysql.connector
import pandas as pd

CONN_PARAMS = {
    'host'    : 'localhost',
    'user'    : 'root',       # ← cambiá si usás otro usuario
    'password': 'v08983523',           # ← tu contraseña de MySQL (string vacío si no tiene)
    'database': 'northwind',
    'charset' : 'utf8mb4',
}

# Abrimos la conexión una sola vez para toda la notebook.
# Al terminar de trabajar, ejecutá la última celda para cerrarla.
conn = mysql.connector.connect(**CONN_PARAMS)
print('✅ Conexión abierta')


def query(sql, params=None):
    """
    Ejecuta una consulta SQL y devuelve un DataFrame.

    Parámetros:
        sql    : string con la consulta SQL
        params : tupla o dict de parámetros (para consultas parametrizadas)

    Ejemplo:
        query("SELECT * FROM Customers WHERE Country = %s", ('Germany',))
    """
    return pd.read_sql(sql, conn, params=params)


✅ Conexión abierta


In [2]:
# Resumen rápido: filas por tabla
resumen = query("""
    SELECT 'Categories'   AS Tabla, COUNT(*) AS Filas FROM Categories  UNION ALL
    SELECT 'Suppliers',              COUNT(*)          FROM Suppliers   UNION ALL
    SELECT 'Products',               COUNT(*)          FROM Products    UNION ALL
    SELECT 'Customers',              COUNT(*)          FROM Customers   UNION ALL
    SELECT 'Employees',              COUNT(*)          FROM Employees   UNION ALL
    SELECT 'Shippers',               COUNT(*)          FROM Shippers    UNION ALL
    SELECT 'Orders',                 COUNT(*)          FROM Orders      UNION ALL
    SELECT 'OrderDetails',           COUNT(*)          FROM OrderDetails
""")
print(resumen.to_string(index=False))

       Tabla  Filas
  Categories      8
   Suppliers     10
    Products     50
   Customers     40
   Employees      0
    Shippers      3
      Orders      0
OrderDetails      0


C:\Users\Alexander\AppData\Local\Temp\ipykernel_2104\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


---
# PARTE 1 — Nivel Básico
## SELECT, WHERE, ORDER BY, LIMIT

Todas las consultas siguen la misma estructura:

```sql
SELECT  columnas
FROM    tabla
WHERE   condición
ORDER BY columna ASC|DESC
LIMIT   n;
```

### 1.1 — Ver todas las categorías de productos

In [3]:
query("SELECT * FROM Categories")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_2104\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,CategoryID,CategoryName,Description
0,1,Beverages,"Soft drinks, coffees, teas, beers and ales"
1,2,Condiments,"Sweet and savory sauces, relishes, spreads and..."
2,3,Confections,"Desserts, candies and sweet breads"
3,4,Dairy Products,Cheeses
4,5,Grains/Cereals,"Breads, crackers, pasta and cereal"
5,6,Meat/Poultry,Prepared meats
6,7,Produce,Dried fruit and bean curd
7,8,Seafood,Seaweed and fish


### 1.2 — Productos con su precio y stock

In [4]:
query("""
    SELECT ProductName,
           UnitPrice,
           UnitsInStock,
           Discontinued
    FROM   Products
    ORDER BY UnitPrice DESC
    LIMIT 10
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_2104\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,ProductName,UnitPrice,UnitsInStock,Discontinued
0,C├┤te de Blaye,263.50,17,0
1,Th├╝ringer Rostbratwurst,123.79,0,1
2,Mishi Kobe Niku,97.00,29,1
3,Sir Rodney's Marmalade,81.00,40,0
4,Carnarvon Tigers,62.50,42,0
5,Ipoh Coffee,46.00,17,0
6,R├Âsti,45.60,26,0
7,R├Âssle Sauerkraut,45.60,26,1
8,Schoggi Schokolade,43.90,49,0
9,Northwoods Cranberry Sauce,40.00,6,0


### 1.3 — Filtrar por condición: productos caros con stock disponible

In [ ]:
query("""
    SELECT ProductName,
           UnitPrice,
           UnitsInStock
    FROM   Products
    WHERE  UnitPrice > 50
      AND  UnitsInStock > 0
      AND  Discontinued = 0
    ORDER BY UnitPrice DESC
""")

### 1.4 — Clientes de un país específico

> 💡 Usamos **consultas parametrizadas** con `%s` para evitar SQL injection.

In [ ]:
pais = 'Germany'

query("""
    SELECT CustomerID,
           CompanyName,
           ContactName,
           City
    FROM   Customers
    WHERE  Country = %s
    ORDER BY CompanyName
""", (pais,))

### 1.5 — Búsqueda con LIKE: productos que contienen una palabra

In [ ]:
query("""
    SELECT ProductName, UnitPrice, UnitsInStock
    FROM   Products
    WHERE  ProductName LIKE '%Cheese%'
       OR  ProductName LIKE '%Queso%'
    ORDER BY ProductName
""")

### 1.6 — Columnas calculadas: total en stock

Podemos crear columnas calculadas directamente en SQL.

In [ ]:
query("""
    SELECT ProductName,
           UnitPrice,
           UnitsInStock,
           ROUND(UnitPrice * UnitsInStock, 2) AS ValorEnStock
    FROM   Products
    WHERE  Discontinued = 0
    ORDER BY ValorEnStock DESC
    LIMIT 10
""")

### 1.7 — Pedidos en un rango de fechas

In [ ]:
query("""
    SELECT OrderID,
           CustomerID,
           OrderDate,
           ShipCountry,
           Freight
    FROM   Orders
    WHERE  OrderDate BETWEEN '2023-10-01' AND '2023-10-31'
    ORDER BY OrderDate
""")

### 1.8 — Productos descontinuados vs activos

In [ ]:
query("""
    SELECT
        CASE Discontinued
            WHEN 0 THEN 'Activo'
            WHEN 1 THEN 'Descontinuado'
        END AS Estado,
        COUNT(*) AS Cantidad
    FROM Products
    GROUP BY Discontinued
""")

---
## ✏️ Ejercicios — Parte 1

**Ejercicio 1.** Listá todos los empleados ordenados por fecha de contratación (más antiguos primero). Mostrá nombre, apellido, cargo y fecha.

**Ejercicio 2.** ¿Cuántos clientes hay en cada país? Mostrá los 10 países con más clientes.

**Ejercicio 3.** Encontrá los productos cuyo stock está por debajo del nivel de reposición (`ReorderLevel`). Mostrá nombre, stock actual y nivel de reposición, ordenados por la diferencia.

**Ejercicio 4.** Listá los pedidos enviados a Brasil. ¿Cuál fue el flete más caro?

**Ejercicio 5.** Mostrá los pedidos del mes de noviembre 2023, incluyendo una columna calculada que indique cuántos días tardó en enviarse (`ShippedDate - OrderDate`).

In [ ]:
# Ejercicio 1


In [ ]:
# Ejercicio 2


In [ ]:
# Ejercicio 3


In [ ]:
# Ejercicio 4


In [ ]:
# Ejercicio 5


---
# PARTE 2 — Nivel Intermedio
## JOIN, GROUP BY, Subconsultas

### Repaso visual de JOINs

```
INNER JOIN          LEFT JOIN          RIGHT JOIN
  A ∩ B               A ∪ (A∩B)          B ∪ (A∩B)
  ██████               ██████████         ██████████
 ███████              ████████████       ████████████
 solo lo que         todo A +            todo B +
 coincide en         lo que coincide     lo que coincide
 ambas tablas        en B                en A
```

### 2.1 — INNER JOIN: productos con su categoría y proveedor

In [ ]:
query("""
    SELECT p.ProductName,
           c.CategoryName,
           s.CompanyName    AS Supplier,
           s.Country        AS SupplierCountry,
           p.UnitPrice
    FROM   Products   p
    JOIN   Categories c ON p.CategoryID  = c.CategoryID
    JOIN   Suppliers  s ON p.SupplierID  = s.SupplierID
    WHERE  p.Discontinued = 0
    ORDER BY c.CategoryName, p.UnitPrice DESC
    LIMIT 15
""")

### 2.2 — JOIN con 4 tablas: detalle completo de pedidos

In [ ]:
query("""
    SELECT o.OrderID,
           o.OrderDate,
           c.CompanyName                                  AS Cliente,
           CONCAT(e.FirstName, ' ', e.LastName)           AS Empleado,
           p.ProductName,
           od.Quantity                                    AS Cantidad,
           od.UnitPrice                                   AS PrecioUnit,
           od.Discount                                    AS Descuento,
           ROUND(od.Quantity * od.UnitPrice * (1 - od.Discount), 2) AS Total
    FROM   Orders      o
    JOIN   Customers   c  ON o.CustomerID  = c.CustomerID
    JOIN   Employees   e  ON o.EmployeeID  = e.EmployeeID
    JOIN   OrderDetails od ON o.OrderID    = od.OrderID
    JOIN   Products    p  ON od.ProductID  = p.ProductID
    ORDER BY o.OrderDate DESC
    LIMIT 15
""")

### 2.3 — GROUP BY: ventas totales por país

`GROUP BY` agrupa filas y permite aplicar **funciones de agregación**: `SUM`, `COUNT`, `AVG`, `MAX`, `MIN`.

In [ ]:
query("""
    SELECT  c.Country,
            COUNT(DISTINCT o.OrderID)                                AS TotalPedidos,
            COUNT(DISTINCT o.CustomerID)                             AS Clientes,
            ROUND(SUM(od.Quantity * od.UnitPrice * (1 - od.Discount)), 2) AS RevenueTotal,
            ROUND(AVG(od.Quantity * od.UnitPrice * (1 - od.Discount)), 2) AS TicketPromedio
    FROM    Orders      o
    JOIN    Customers   c  ON o.CustomerID  = c.CustomerID
    JOIN    OrderDetails od ON o.OrderID    = od.OrderID
    GROUP BY c.Country
    ORDER BY RevenueTotal DESC
""")

### 2.4 — HAVING: filtrar grupos

`HAVING` es como `WHERE` pero **después de agrupar**. Solo permite condiciones sobre columnas agregadas.

In [ ]:
query("""
    SELECT  c.Country,
            COUNT(DISTINCT o.OrderID) AS TotalPedidos,
            ROUND(SUM(od.Quantity * od.UnitPrice * (1 - od.Discount)), 2) AS Revenue
    FROM    Orders       o
    JOIN    Customers    c  ON o.CustomerID  = c.CustomerID
    JOIN    OrderDetails od ON o.OrderID     = od.OrderID
    GROUP BY c.Country
    HAVING  COUNT(DISTINCT o.OrderID) >= 5    -- solo países con 5+ pedidos
    ORDER BY Revenue DESC
""")

### 2.5 — Ranking de empleados por ventas

In [ ]:
query("""
    SELECT  CONCAT(e.FirstName, ' ', e.LastName)   AS Empleado,
            e.Title,
            COUNT(DISTINCT o.OrderID)              AS TotalPedidos,
            ROUND(SUM(od.Quantity * od.UnitPrice * (1 - od.Discount)), 2) AS VentasTotal
    FROM    Employees    e
    JOIN    Orders       o  ON e.EmployeeID  = o.EmployeeID
    JOIN    OrderDetails od ON o.OrderID     = od.OrderID
    GROUP BY e.EmployeeID, e.FirstName, e.LastName, e.Title
    ORDER BY VentasTotal DESC
""")

### 2.6 — Ventas por categoría y mes

In [ ]:
query("""
    SELECT  DATE_FORMAT(o.OrderDate, '%Y-%m') AS Mes,
            cat.CategoryName,
            ROUND(SUM(od.Quantity * od.UnitPrice * (1 - od.Discount)), 2) AS Revenue
    FROM    Orders       o
    JOIN    OrderDetails od  ON o.OrderID    = od.OrderID
    JOIN    Products     p   ON od.ProductID = p.ProductID
    JOIN    Categories   cat ON p.CategoryID = cat.CategoryID
    GROUP BY Mes, cat.CategoryName
    ORDER BY Mes, Revenue DESC
""")

### 2.7 — LEFT JOIN: clientes que nunca compraron

In [ ]:
query("""
    SELECT  c.CustomerID,
            c.CompanyName,
            c.Country,
            COUNT(o.OrderID) AS TotalPedidos
    FROM    Customers c
    LEFT JOIN Orders  o ON c.CustomerID = o.CustomerID
    GROUP BY c.CustomerID, c.CompanyName, c.Country
    HAVING  TotalPedidos = 0
""")

### 2.8 — Subconsulta: productos con precio mayor al promedio de su categoría

In [ ]:
query("""
    SELECT  p.ProductName,
            cat.CategoryName,
            p.UnitPrice,
            ROUND(prom.PromedioCat, 2) AS PromedioDeLaCategoria
    FROM    Products    p
    JOIN    Categories  cat ON p.CategoryID = cat.CategoryID
    JOIN (
        -- Subconsulta: precio promedio por categoría
        SELECT  CategoryID,
                AVG(UnitPrice) AS PromedioCat
        FROM    Products
        WHERE   Discontinued = 0
        GROUP BY CategoryID
    ) prom ON p.CategoryID = prom.CategoryID
    WHERE   p.UnitPrice > prom.PromedioCat
      AND   p.Discontinued = 0
    ORDER BY cat.CategoryName, p.UnitPrice DESC
""")

### 2.9 — Subconsulta correlacionada: el pedido más grande de cada cliente

In [ ]:
query("""
    SELECT  c.CompanyName,
            c.Country,
            o.OrderID,
            o.OrderDate,
            ROUND(SUM(od.Quantity * od.UnitPrice * (1 - od.Discount)), 2) AS TotalPedido
    FROM    Orders       o
    JOIN    Customers    c   ON o.CustomerID  = c.CustomerID
    JOIN    OrderDetails od  ON o.OrderID     = od.OrderID
    GROUP BY o.OrderID, c.CompanyName, c.Country, o.OrderDate
    HAVING  TotalPedido = (
        -- Subconsulta: máximo revenue para ese cliente
        SELECT MAX(sub_total)
        FROM (
            SELECT  o2.CustomerID,
                    SUM(od2.Quantity * od2.UnitPrice * (1 - od2.Discount)) AS sub_total
            FROM    Orders       o2
            JOIN    OrderDetails od2 ON o2.OrderID = od2.OrderID
            WHERE   o2.CustomerID = o.CustomerID
            GROUP BY o2.OrderID
        ) t
    )
    ORDER BY TotalPedido DESC
    LIMIT 10
""")

### 2.10 — Usar la vista `vw_orden_detalle`

El script SQL creó una vista que ya resuelve todos los JOINs. Las vistas simplifican consultas complejas.

In [ ]:
# Vista completa
query("SELECT * FROM vw_orden_detalle LIMIT 10")

In [ ]:
# Ahora es muy simple consultar sobre la vista
query("""
    SELECT  CustomerCountry,
            CategoryName,
            ROUND(SUM(LineTotal), 2) AS Revenue
    FROM    vw_orden_detalle
    GROUP BY CustomerCountry, CategoryName
    ORDER BY Revenue DESC
    LIMIT 15
""")

### 2.11 — Integración Python + SQL: análisis dinámico

In [ ]:
# Traemos los datos y hacemos el análisis final en Pandas
df_ventas = query("""
    SELECT  DATE_FORMAT(OrderDate, '%Y-%m') AS Mes,
            ROUND(SUM(LineTotal), 2)        AS Revenue,
            COUNT(DISTINCT OrderID)         AS Pedidos
    FROM    vw_orden_detalle
    GROUP BY Mes
    ORDER BY Mes
""")

df_ventas['TicketPromedio'] = (df_ventas['Revenue'] / df_ventas['Pedidos']).round(2)
df_ventas['RevenuePct']     = (df_ventas['Revenue'] / df_ventas['Revenue'].sum() * 100).round(1)

print("── Evolución mensual de ventas ──")
print(df_ventas.to_string(index=False))

print(f"\n── Estadísticas del ticket promedio mensual ──")
print(f"Media   : £{df_ventas['TicketPromedio'].mean():.2f}")
print(f"Mediana : £{df_ventas['TicketPromedio'].median():.2f}")
print(f"Desvío  : £{df_ventas['TicketPromedio'].std():.2f}")

---
## ✏️ Ejercicios — Parte 2

**Ejercicio 6.** Mostrá los 5 productos más vendidos (por cantidad total). Incluí nombre del producto, categoría y cantidad total vendida.

**Ejercicio 7.** ¿Qué proveedor genera más revenue en total? Calculá el revenue por proveedor, ordenado de mayor a menor.

**Ejercicio 8.** Calculá el tiempo promedio de envío (días entre `OrderDate` y `ShippedDate`) por empresa de envío (`Shippers`). ¿Cuál es más rápida?

**Ejercicio 9.** Encontrá los clientes que hicieron **más de 3 pedidos** y cuyo **revenue total supera £1000**. Mostrá cliente, país, cantidad de pedidos y revenue total.

**Ejercicio 10.** Usando la vista `vw_orden_detalle`, encontrá el empleado con mayor revenue en cada país de envío. *(Pista: subconsulta o CTE)*

In [ ]:
# Ejercicio 6


In [ ]:
# Ejercicio 7


In [ ]:
# Ejercicio 8


In [ ]:
# Ejercicio 9


In [ ]:
# Ejercicio 10


---
## 🗂️ Referencia rápida

```sql
-- Estructura base
SELECT  col1, col2, funcion(col3) AS alias
FROM    TablaA a
JOIN    TablaB b  ON a.id = b.id
WHERE   condicion
GROUP BY col1, col2
HAVING  condicion_sobre_agregados
ORDER BY alias DESC
LIMIT   n;

-- Funciones de agregación
COUNT(*)          -- cuenta filas
COUNT(DISTINCT x) -- cuenta valores únicos
SUM(x)            -- suma
AVG(x)            -- promedio
MAX(x) / MIN(x)   -- máximo / mínimo

-- Funciones de fecha
DATE_FORMAT(fecha, '%Y-%m')   -- año-mes
DATEDIFF(fecha1, fecha2)      -- diferencia en días
MONTH(fecha) / YEAR(fecha)    -- mes / año

-- Tipos de JOIN
INNER JOIN  -- solo filas que coinciden en ambas tablas
LEFT JOIN   -- todas las filas de la tabla izquierda
RIGHT JOIN  -- todas las filas de la tabla derecha
```

In [ ]:
# ── Cerrá la conexión cuando termines de trabajar ──
conn.close()
print('🔒 Conexión cerrada')